In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

In [2]:
DATA_PATH = 'Data/train.csv'

def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df = df.dropna(subset=['target']).reset_index(drop=True)
    return df

df = load_data()

In [7]:
def train_val_split(df, val_size=0.2):

    dates = np.sort(df['date_id'].unique())
    n_val = int(len(dates) * val_size)
    cutoff_date = dates[-n_val]

    train_df = df[df['date_id'] < cutoff_date].reset_index(drop=True)
    val_df = df[df['date_id'] >= cutoff_date].reset_index(drop=True)

    print(f"Train set: {train_df.date_id.nunique()} samples, Validation set: {val_df.date_id.nunique()} samples")

    return train_df, val_df

train_df, val_df = train_val_split(df, val_size=0.2)

Train set: 385 samples, Validation set: 96 samples


In [8]:
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

In [9]:
baseline_zero = mae(val_df['target'], np.zeros(len(val_df)))
print(f"Zero-prediction MAE on validation: {baseline_zero:.4f}")

Zero-prediction MAE on validation: 6.0601


## Linear Model

In [10]:
FEATURES = [
    'imbalance_buy_sell_flag', 'signed_imb', 'imb_ratio', 'book_imb',
    'near_minus_wap', 'ref_minus_wap', 'spread',
    'imbalance_size', 'matched_size', 'bid_size', 'ask_size',
    'wap', 'reference_price', 'seconds_in_bucket', 'stock_id',
]

def build_features(df):
    df = df.copy()
    df['signed_imb']     = df['imbalance_size'] * df['imbalance_buy_sell_flag']
    df['imb_ratio']      = df['signed_imb'] / df['matched_size']
    df['book_imb']       = (df['bid_size'] - df['ask_size']) / (df['bid_size'] + df['ask_size'])
    df['near_minus_wap'] = df['near_price'] - df['wap']       # NaN before 300s — left as-is for LGBM
    df['ref_minus_wap']  = df['reference_price'] - df['wap']
    df['spread']         = df['ask_price'] - df['bid_price']
    return df

train_fe = build_features(train_df)
val_fe   = build_features(val_df)

y_train, y_val = train_fe['target'], val_fe['target']

In [11]:
train_fe.columns

Index(['stock_id', 'date_id', 'seconds_in_bucket', 'imbalance_size',
       'imbalance_buy_sell_flag', 'reference_price', 'matched_size',
       'far_price', 'near_price', 'bid_price', 'bid_size', 'ask_price',
       'ask_size', 'wap', 'target', 'time_id', 'row_id', 'signed_imb',
       'imb_ratio', 'book_imb', 'near_minus_wap', 'ref_minus_wap', 'spread'],
      dtype='object')

In [12]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [14]:
linear_features = ['stock_id', 'seconds_in_bucket', 'imbalance_size','imbalance_buy_sell_flag', 'signed_imb', 'imb_ratio', 'book_imb', 'near_minus_wap', 'ref_minus_wap', 'spread', 'matched_size', 'bid_size', 'ask_size', 'wap', 'reference_price']

def clean_for_linear(df, features):
    df = df.copy()

    X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

    return X

X_train_lin = clean_for_linear(train_fe, linear_features)
X_val_lin   = clean_for_linear(val_fe,   linear_features)



In [15]:
scaler = StandardScaler()
X_train_lin_scaled = scaler.fit_transform(X_train_lin)
X_val_lin_scaled   = scaler.transform(X_val_lin)

In [16]:
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_train_lin_scaled, y_train)
y_pred_ridge = ridge.predict(X_val_lin_scaled)
pred_mae = mae(y_val, y_pred_ridge)
print(f"Ridge Regression MAE on validation: {pred_mae:.4f}")

Ridge Regression MAE on validation: 6.0306


In [17]:
# get weights of the ridge regression model
ridge_weights = pd.Series(ridge.coef_, index=X_train_lin.columns).sort_values(key=abs, ascending=False)
ridge_weights

ref_minus_wap              0.941320
book_imb                  -0.825049
imbalance_buy_sell_flag   -0.221022
near_minus_wap             0.180184
signed_imb                 0.178185
ask_size                  -0.133933
bid_size                   0.129111
wap                       -0.116686
reference_price           -0.046010
spread                     0.042631
seconds_in_bucket         -0.031629
imb_ratio                  0.024192
imbalance_size             0.020684
matched_size               0.009536
stock_id                   0.008006
dtype: float64

## LightGBM

In [19]:
import lightgbm as lgb

In [20]:
X_train = train_fe[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = val_fe[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)

In [22]:
model = lgb.LGBMRegressor(
    objective='mae',          # optimize what we're scored on
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=128,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    min_child_samples=100,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='mae',
    categorical_feature=['stock_id'],
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True)]
)

pred_lgb = model.predict(X_val)
lgb_mae = mae(y_val, pred_lgb)
print(f"LightGBM MAE on validation: {lgb_mae:.4f}")

/Users/vivekn/anaconda3/envs/dev/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015169 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3318
[LightGBM] [Info] Number of data points in the train set: 4181948, number of used features: 15
[LightGBM] [Info] Start training from score -0.069737
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[278]	valid_0's l1: 5.95528
LightGBM MAE on validation: 5.9553
